### Functional Manner to Compute Results Table - Vector Error Correction Model

In [1]:
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
def prep_event_data(cme_df, pm_df, kalshi_df=None):
    df_cme = cme_df[["time", "prob_no_change"]].copy()
    df_cme["time"] = pd.to_datetime(df_cme["time"])
    df_cme = df_cme.rename(columns={"time": "timestamp"}).set_index("timestamp")
    cme_min = df_cme.resample("1Min").last()

    df_pm = pm_df[["datetime", "yes_price"]].copy()
    df_pm["datetime"] = pd.to_datetime(df_pm["datetime"])
    df_pm = df_pm.rename(columns={"datetime": "timestamp", "yes_price": "yes_price_PM"}).set_index("timestamp")
    pm_min = df_pm.resample("1Min").last().shift(1)

    merged_df = pd.merge(cme_min, pm_min, left_index=True, right_index=True, how="outer")

    if kalshi_df is not None:
        df_k = kalshi_df[["created_time", "yes_price"]].copy()
        df_k["created_time"] = pd.to_datetime(df_k["created_time"], format='mixed')
        df_k = df_k.rename(columns={"created_time": "timestamp", "yes_price": "yes_price_KAL"}).set_index("timestamp")
        kalshi_min = df_k.resample("1Min").last().shift(1)
        merged_df = pd.merge(merged_df, kalshi_min, left_index=True, right_index=True, how="outer")

    merged_df = merged_df.sort_index().ffill().dropna()

    window_mins = 3 * 24 * 60
    rolling_var = merged_df["prob_no_change"].rolling(window=window_mins, min_periods=window_mins).var()
    is_active = rolling_var > 1e-6
    
    if is_active.any():
        first_active_t = merged_df[is_active].index.min()
        start_t = max(merged_df.index.min(), first_active_t - pd.Timedelta(minutes=window_mins))
        merged_df = merged_df.loc[start_t:]
    
    return merged_df

In [3]:
def plot_event_probabilities(df_clean, event_name="Event"):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.set_xlabel("Timestamp", fontsize=12, labelpad=10)
    ax1.set_ylabel("Implied Probability", fontsize=12)

    # Plotting CME's prob_no_change
    ax1.plot(df_clean.index, df_clean["prob_no_change"], color="#1f77b4", linewidth=1.5, label="CME (0bp)")
    
    if "yes_price_PM" in df_clean.columns:
        ax1.plot(df_clean.index, df_clean["yes_price_PM"], color="#ff7f0e", linewidth=1.5, label="Polymarket")
    if "yes_price_KAL" in df_clean.columns:
        ax1.plot(df_clean.index, df_clean["yes_price_KAL"], color="#2ca02c", linewidth=1.5, label="Kalshi")

    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend(loc="upper left", fontsize=11)
    plt.suptitle(f"{event_name}", fontsize=14, fontweight="bold", y=0.96)
    plt.show()

In [4]:
def test_vecm_prerequisites(df_clean, event_date, contract_type, mkt_1, mkt_2, lags=[1, 5, 30, 60]):
    
    col_map = {"CME": "prob_no_change","PM": "yes_price_PM","KAL": "yes_price_KAL"}
    src_col, tgt_col = col_map[mkt_1], col_map[mkt_2]
    
    def get_adf_pvalue(series):
        return round(adfuller(series.dropna())[1], 4)

    adf_src_raw = get_adf_pvalue(df_clean[src_col])
    adf_tgt_raw = get_adf_pvalue(df_clean[tgt_col])
    adf_src_diff = get_adf_pvalue(df_clean[src_col].diff())
    adf_tgt_diff = get_adf_pvalue(df_clean[tgt_col].diff())
    
    is_raw_stationary = (adf_src_raw < 0.05) or (adf_tgt_raw < 0.05)
    is_firstdiff_stationary = (adf_src_diff < 0.05) or (adf_tgt_diff < 0.05)
    
    johansen = coint_johansen(df_clean[[src_col, tgt_col]], det_order=0, k_ar_diff=1)
    is_cointegrated = johansen.lr1[0] > johansen.cvt[0, 1]

    results_list = []
    for lag in lags:
        results_list.append({
            "event_date": event_date,
            "contract": contract_type,
            "source_market": mkt_1,
            "target_market": mkt_2,
            "lag_minute": lag,
            "raw_stationarity_source_pvalue": adf_src_raw,
            "raw_stationarity_target_pvalue": adf_tgt_raw,
            "is_raw_stationary": is_raw_stationary,
            "first_differenced_stationarity_source_pvalue": adf_src_diff,
            "first_differenced_stationarity_target_pvalue": adf_tgt_diff,
            "is_firstdiff_stationary": is_firstdiff_stationary,
            "johansen_trace_stat": round(johansen.lr1[0], 2),
            "is_cointegrated": is_cointegrated
        })
    return results_list

In [5]:
def run_vecm_pipeline(df_clean, event_date, contract_type, mkt_1, mkt_2, lags=[1, 5, 30, 60]):
    col_map = {
        "CME": "prob_no_change",
        "PM": "yes_price_PM",
        "KAL": "yes_price_KAL"
    }
    
    target_name = col_map[mkt_2]
    source_name = col_map[mkt_1]
    
    y1 = df_clean[target_name]
    y2 = df_clean[source_name]
    
    X_long_run = sm.add_constant(y2)
    long_run_model = sm.OLS(y1, X_long_run).fit()
    
    mu = long_run_model.params["const"]
    beta = long_run_model.params[source_name]
    
    ect = y1 - mu - (beta * y2)
    
    results_list = []

    for lag in lags:
        df_reg = pd.DataFrame({
            "delta_y1": y1.diff(),
            "ect_lagged": ect.shift(lag)
        }).dropna()
        
        error_correction_model = sm.OLS(df_reg["delta_y1"], df_reg["ect_lagged"]).fit()
        
        p_val = error_correction_model.pvalues.iloc[0]
        
        results_list.append({
            "event_date": event_date,
            "contract": contract_type,
            "source_market": mkt_1,
            "target_market": mkt_2,
            "lag_minute": lag,
            "VECM_coefficient": round(error_correction_model.params.iloc[0], 6),
            "p_value": round(p_val, 6),
            "is_significant": p_val < 0.05
        })
        
    return results_list

### Vector Error Correction Model Results for All Events for 2025 (0bp cut Market)

In [6]:
# Configuration (2025 Event Suite)
event_config = {
    "2025-01-29": {"month_code": "JAN"},
    "2025-03-19": {"month_code": "MAR"},
    "2025-05-07": {"month_code": "MAY"},
    "2025-06-18": {"month_code": "JUN"},
    "2025-07-30": {"month_code": "JUL"},
    "2025-09-17": {"month_code": "SEP"},
    "2025-10-29": {"month_code": "OCT"},
    "2025-12-10": {"month_code": "DEC"}
}

market_pairs = [("CME", "PM"), ("CME", "KAL"), ("PM", "KAL")]
all_prereq_results = []

# Main Processing Loop
for slug, config in event_config.items():
    month = config["month_code"]
    print(f"\n{'#'*60}\nPROCESSING PREREQUISITES: {slug} ({month})\n{'#'*60}")
    
    try:
        # Load data
        cme_raw = pd.read_csv(f"../../data/processed/CME_implied_probabilities/CME_IMP_{month}_2025.csv")
        pm_raw = pd.read_csv(f"../../data/raw/Polymarket/data_0bp_cut/PM_{month}_2025.csv")
        kal_raw = pd.read_csv(f"../../data/raw/Kalshi/data_0bp_cut/kalshi_{month}_2025.csv")

        
        # Prep data
        clean_df = prep_event_data(cme_df=cme_raw, pm_df=pm_raw, kalshi_df=kal_raw)
        
        # Run Prerequisite suite for all pairs
        for mkt_a, mkt_b in market_pairs:
            print(f"  Testing {mkt_a} <-> {mkt_b}...")
            
            # Run test once per pair
            pair_results = test_vecm_prerequisites(
                df_clean=clean_df, 
                event_date=slug, 
                contract_type="0bp", 
                mkt_1=mkt_a, 
                mkt_2=mkt_b,
                lags=[1, 5, 30, 60]
            )
            
            # Add original direction
            all_prereq_results.extend(pair_results)
            
            # Add switched direction
            switched_results = []
            for row in pair_results:
                row_copy = row.copy()
                row_copy["source_market"] = mkt_b
                row_copy["target_market"] = mkt_a
                switched_results.append(row_copy)
            all_prereq_results.extend(switched_results)
            
    except Exception as e:
        print(f"X Error processing {slug}: {e}")


prereq_master_df = pd.DataFrame(all_prereq_results)


############################################################
PROCESSING PREREQUISITES: 2025-01-29 (JAN)
############################################################
  Testing CME <-> PM...
  Testing CME <-> KAL...
  Testing PM <-> KAL...

############################################################
PROCESSING PREREQUISITES: 2025-03-19 (MAR)
############################################################
  Testing CME <-> PM...
  Testing CME <-> KAL...
  Testing PM <-> KAL...

############################################################
PROCESSING PREREQUISITES: 2025-05-07 (MAY)
############################################################
  Testing CME <-> PM...
  Testing CME <-> KAL...
  Testing PM <-> KAL...

############################################################
PROCESSING PREREQUISITES: 2025-06-18 (JUN)
############################################################
  Testing CME <-> PM...
  Testing CME <-> KAL...
  Testing PM <-> KAL...

###########################################

In [7]:
event_config = {
    "2025-01-29": {"month_code": "JAN"},
    "2025-03-19": {"month_code": "MAR"},
    "2025-05-07": {"month_code": "MAY"},
    "2025-06-18": {"month_code": "JUN"},
    "2025-07-30": {"month_code": "JUL"},
    "2025-09-17": {"month_code": "SEP"},
    "2025-10-29": {"month_code": "OCT"},
    "2025-12-10": {"month_code": "DEC"}
}

market_pairs = [("CME", "PM"), ("CME", "KAL"), ("PM", "KAL")]
all_vecm_results = []

for slug, config in event_config.items():
    month = config["month_code"]
    print(f"\n{'#'*60}\nPROCESSING VECM: {slug} ({month})\n{'#'*60}")
    
    try:

        cme_raw = pd.read_csv(f"../../data/processed/CME_implied_probabilities/CME_IMP_{month}_2025.csv")
        pm_raw = pd.read_csv(f"../../data/raw/Polymarket/data_0bp_cut/PM_{month}_2025.csv")
        kal_raw = pd.read_csv(f"../../data/raw/Kalshi/data_0bp_cut/kalshi_{month}_2025.csv")
        
        clean_df = prep_event_data(cme_df=cme_raw, pm_df=pm_raw, kalshi_df=kal_raw)
        
        for mkt_a, mkt_b in market_pairs:
            for src, tgt in [(mkt_a, mkt_b), (mkt_b, mkt_a)]:
                print(f"  Running VECM: {src} -> {tgt}...")
                
                pair_results = run_vecm_pipeline(
                    df_clean=clean_df, 
                    event_date=slug, 
                    contract_type="0bp", 
                    mkt_1=src, 
                    mkt_2=tgt,
                    lags=[1, 5, 30, 60]
                )
                all_vecm_results.extend(pair_results)
                
    except Exception as e:
        print(f"X Error processing VECM for {slug}: {e}")

vecm_master_df = pd.DataFrame(all_vecm_results)



############################################################
PROCESSING VECM: 2025-01-29 (JAN)
############################################################
  Running VECM: CME -> PM...
  Running VECM: PM -> CME...
  Running VECM: CME -> KAL...
  Running VECM: KAL -> CME...
  Running VECM: PM -> KAL...
  Running VECM: KAL -> PM...

############################################################
PROCESSING VECM: 2025-03-19 (MAR)
############################################################
  Running VECM: CME -> PM...
  Running VECM: PM -> CME...
  Running VECM: CME -> KAL...
  Running VECM: KAL -> CME...
  Running VECM: PM -> KAL...
  Running VECM: KAL -> PM...

############################################################
PROCESSING VECM: 2025-05-07 (MAY)
############################################################
  Running VECM: CME -> PM...
  Running VECM: PM -> CME...
  Running VECM: CME -> KAL...
  Running VECM: KAL -> CME...
  Running VECM: PM -> KAL...
  Running VECM: KAL -> PM...



In [8]:
vecm_master_df

,event_date,contract,source_market,target_market,lag_minute,VECM_coefficient,p_value,is_significant
0,2025-01-29,0bp,CME,PM,1,-0.007279,0.000000,True
1,2025-01-29,0bp,CME,PM,5,-0.002257,0.000002,True
2,2025-01-29,0bp,CME,PM,30,-0.000706,0.140337,False
3,2025-01-29,0bp,CME,PM,60,-0.000464,0.332800,False
4,2025-01-29,0bp,PM,CME,1,-0.001775,0.000000,True
...,...,...,...,...,...,...,...,...
187,2025-12-10,0bp,PM,KAL,60,-0.000387,0.081566,False
188,2025-12-10,0bp,KAL,PM,1,-0.006048,0.000000,True
189,2025-12-10,0bp,KAL,PM,5,-0.002101,0.000000,True
190,2025-12-10,0bp,KAL,PM,30,-0.000768,0.002159,True


In [9]:
merge_keys = ['event_date', 'contract', 'source_market', 'target_market', 'lag_minute']
cols_to_keep = merge_keys + ['is_raw_stationary', 'is_firstdiff_stationary', 'is_cointegrated']

final_analysis_df = pd.merge(
    vecm_master_df, 
    prereq_master_df[cols_to_keep], 
    on=merge_keys, 
    how='left'
)

In [10]:
# Viewing the results as dataframe 
final_analysis_df

,event_date,contract,source_market,target_market,lag_minute,VECM_coefficient,p_value,is_significant,is_raw_stationary,is_firstdiff_stationary,is_cointegrated
0,2025-01-29,0bp,CME,PM,1,-0.007279,0.000000,True,True,True,True
1,2025-01-29,0bp,CME,PM,5,-0.002257,0.000002,True,True,True,True
2,2025-01-29,0bp,CME,PM,30,-0.000706,0.140337,False,True,True,True
3,2025-01-29,0bp,CME,PM,60,-0.000464,0.332800,False,True,True,True
4,2025-01-29,0bp,PM,CME,1,-0.001775,0.000000,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
187,2025-12-10,0bp,PM,KAL,60,-0.000387,0.081566,False,False,True,True
188,2025-12-10,0bp,KAL,PM,1,-0.006048,0.000000,True,False,True,True
189,2025-12-10,0bp,KAL,PM,5,-0.002101,0.000000,True,False,True,True
190,2025-12-10,0bp,KAL,PM,30,-0.000768,0.002159,True,False,True,True


In [ ]:
# Saving the dataframe in the defined directory 
final_analysis_df.to_csv("../../results/vector_error_correction_model/vector_error_correction_mode_0bp_cut_2025.csv", index = False)